# Test the production agent with a sample summary

This notebook uses the basic `construct_prompt(summary, ticker)` path from production, then calls the configured model without submitting anything to the competition API. Edit the sample ticker and summary below; the ticker selects its mapped industry and cached dossier automatically.

This is intended for quickly checking the rendered prompt and the model's prediction shape.

In [ ]:
from __future__ import annotations

from pathlib import Path
import json
import os
import re
import sys

from dotenv import load_dotenv
from litellm import completion, completion_cost, token_counter

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == 'notebooks':
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))
load_dotenv(REPO_ROOT / '.env', override=False)

from predict import (
    MODEL, LLM_MAX_RETRIES, LLM_TIMEOUT_SECONDS, Prediction,
    _normalize_percentile, _parse_prediction, _required_api_key,
)
from prompt_construction import construct_prompt, get_dossier

print('Model:', MODEL)
print('Provider key loaded:', bool(os.getenv(_required_api_key(MODEL))))


## 1. Choose a sample ticker and summary

Use a ticker with a file in `knowledge/dossier` to exercise dossier injection. Replace the summary with any event text you want to test.

In [ ]:
TICKER = 'PLUG'
SAMPLE_SUMMARY = (
    '- Quarterly revenue exceeded consensus by approximately 4%.\n'
    '- Adjusted EPS beat consensus, helped by gross-margin expansion.\n'
    '- Management raised full-year revenue guidance but maintained EPS guidance.\n'
    '- Demand remained healthy while management noted higher second-half input costs.'
)

ticker = TICKER.strip().upper()
dossier_text = get_dossier(ticker)
print('Ticker:', ticker)
print('Dossier used by prompt construction:')
print(dossier_text)


## 2. Construct the real production prompt

This calls the same basic constructor used by `predict.py`. No notebook-specific prompt assembly or random industry override is involved.

In [ ]:
user_prompt = construct_prompt(SAMPLE_SUMMARY, ticker)
messages = [
    {'role': 'system', 'content': user_prompt},
    {'role': 'user', 'content': 'Return the prediction as JSON using the requested schema.'},
]
try:
    estimated_input_tokens = token_counter(model=MODEL, messages=messages)
except Exception:
    estimated_input_tokens = None

print(f'Prompt characters: {len(user_prompt):,}')
print('Estimated input tokens:', estimated_input_tokens)
print('Unresolved placeholders:', sorted(set(re.findall(r'\{[a-z_]+\}', user_prompt))))
print(user_prompt)


## 3. Sanity-check the rendered prompt

Confirm the editable summary reached the prompt and that required placeholders were resolved.

In [ ]:
assert SAMPLE_SUMMARY in user_prompt
required_placeholders = r'\{(?:event_bullets|core_directive|industry_rules|dossier)\}'
assert not re.findall(required_placeholders, user_prompt)
print('Prompt is ready for the model.')


## 4. Call the model and validate the production response

This is the only cell that incurs model cost. Validation uses `predict.Prediction`, so only the percentile is required and any current optional metadata is accepted. Nothing is submitted.

In [ ]:
provider_key_name = _required_api_key(MODEL)
if not os.getenv(provider_key_name):
    raise RuntimeError(f'{provider_key_name} is not set; add it to .env before running the paid model cell')

response = completion(
    model=MODEL, messages=messages, response_format={'type': 'json_object'},
    temperature=0, timeout=LLM_TIMEOUT_SECONDS, num_retries=LLM_MAX_RETRIES,
)
raw_content = response.choices[0].message.content
prediction = _parse_prediction(raw_content)
prediction.predicted_percentile = _normalize_percentile(prediction.predicted_percentile)

usage = getattr(response, 'usage', None)
usage_data = usage.model_dump() if hasattr(usage, 'model_dump') else dict(usage or {})
try:
    cost_usd = completion_cost(completion_response=response)
except Exception:
    cost_usd = getattr(response, '_hidden_params', {}).get('response_cost')

print('Validated JSON response:')
print(prediction.model_dump_json(indent=2))
print('Usage:', json.dumps(usage_data, indent=2, default=str))
print('Estimated response cost (USD):', cost_usd if cost_usd is not None else 'unavailable for this provider/model')
prediction
